In [1]:
## Model

'''
So what I have understood is that I need to take the hidden representations of the LSTM and then use them to get the weights

Input > LSTM > Hidden Representation > Softmax Layer > Weights > Calculate Sharpe (Loss) > Train the Model

# So for simplicity, I will choose 5 stocks to invest in instead of using a 50 stocks and choosing them
the stocks will be namely > TATASTEEL.NS, SUNPHARMA.NS, RELIANCE.NS, INFY.NS, TATACONSUM.NS

'''

'\nSo what I have understood is that I need to take the hidden representations of the LSTM and then use them to get the weights\n\nInput > LSTM > Hidden Representation > Softmax Layer > Weights > Calculate Sharpe (Loss) > Train the Model\n\n# So for simplicity, I will choose 5 stocks to invest in instead of using a 50 stocks and choosing them\nthe stocks will be namely > TATASTEEL.NS, SUNPHARMA.NS, RELIANCE.NS, INFY.NS, TATACONSUM.NS\n\n'

In [2]:
import pandas as pd
import numpy as np
import yaml
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
path = os.environ['PROJECT_FOLDER']

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
dataset = pd.read_csv(os.path.join(path,'data','final','Dataset.csv'))
stocks = pd.read_csv(os.path.join(path,'data','final','Stocks.csv'))

dataset.set_index('Date',inplace=True)
stocks.set_index('Date',inplace=True)

stocks = stocks[['TATASTEEL.NS_7DReturn','SUNPHARMA.NS_7DReturn','RELIANCE.NS_7DReturn','INFY.NS_7DReturn','TATACONSUM.NS_7DReturn']]

In [41]:
stocks.dropna(axis=0)

,TATASTEEL.NS_7DReturn,SUNPHARMA.NS_7DReturn,RELIANCE.NS_7DReturn,INFY.NS_7DReturn,TATACONSUM.NS_7DReturn
Date,,,,,
2021-07-26,8.713802,14.216191,3.856454,3.182425,-0.181885
2021-07-27,4.677649,14.161084,2.537607,2.777775,3.503773
2021-07-28,-3.181139,12.204837,1.164005,2.863346,1.817094
2021-07-29,-4.232028,2.506618,2.591742,4.144661,2.069443
2021-08-02,2.000431,-2.031974,1.089691,2.078288,0.859754
...,...,...,...,...,...
2026-06-04,-3.253828,1.038043,2.927967,-2.434510,0.000000
2026-06-08,0.726100,2.574483,4.640721,-2.406306,0.442840
2026-06-09,1.883061,2.905286,4.027641,-8.198727,0.315884


In [42]:
dataset

,GC=F_HL,GC=F_OC,GC=F_Volatility,GC=F_Theta,SI=F_HL,SI=F_OC,SI=F_Volatility,SI=F_Theta,CL=F_HL,CL=F_OC,...,^FCHI_Volatility,^FCHI_Theta,^VIX_HL,^VIX_OC,^VIX_Volatility,^VIX_Theta,^INDIAVIX_HL,^INDIAVIX_OC,^INDIAVIX_Volatility,^INDIAVIX_Theta
Date,,,,,,,,,,,,,,,,,,,,,
2021-07-26,-0.398094,-0.199807,-0.387302,-1.093121,-0.366648,0.016640,-0.356700,-0.011802,-0.144314,-0.150827,...,-0.010667,-1.080627,0.149315,-0.696332,0.150351,-0.939089,-0.330258,0.327226,-0.316902,1.367360
2021-07-27,-0.445094,-0.221287,-0.407848,-1.372308,0.346875,-0.785101,0.407842,-1.365747,-0.498939,-0.328294,...,0.584027,1.356693,0.335931,0.585658,0.225277,0.898915,0.302109,0.192966,0.238089,0.317603
2021-07-28,-0.624530,0.004308,-0.617556,-0.030073,-0.218983,0.187433,-0.196388,1.368811,-0.699129,0.298750,...,-0.608418,-0.758356,0.285036,-0.703338,0.253656,-0.836949,-0.365939,-0.310509,-0.349361,-1.345179
2021-07-29,0.507636,1.325668,0.659063,1.248648,0.552027,1.038574,0.623284,1.332899,-0.401705,0.730586,...,-0.375896,0.146879,-0.438805,-0.079855,-0.479371,-0.339178,-0.387745,-0.062967,-0.405118,-0.286023
2021-08-02,-0.030681,0.310475,-0.087138,0.634994,-0.246038,0.167669,-0.220672,1.442240,0.730801,-1.558726,...,-0.024471,0.939834,0.160626,0.977961,0.212797,1.498324,-0.227176,0.394356,-0.218231,1.235753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-04,2.045673,1.529777,1.806236,0.698820,0.884556,0.989177,0.817746,1.016652,1.073989,-1.594217,...,0.402691,-1.178046,0.013595,-0.584246,0.011899,-0.919892,-0.036871,-0.041989,-0.080682,-0.032899
2026-06-08,1.780780,0.632745,1.412665,0.320090,1.820133,1.392874,1.595135,0.850389,1.691721,-0.996748,...,0.932300,0.393674,0.460340,-0.892484,0.446763,-0.910810,-0.314399,-0.608399,-0.231355,-1.950241
2026-06-09,3.831496,-3.906050,3.873221,-1.071484,3.933660,-4.402805,4.079935,-1.284120,1.989150,-1.813094,...,1.405513,-0.978258,2.388695,1.244167,2.026823,0.802640,-0.425410,0.020945,-0.441672,0.195190


In [6]:
dataset = MyDataset(X=dataset,y=stocks)

Epoch 1: -0.1636
Epoch 2: -0.2029
Epoch 3: -0.2325
Epoch 4: -0.2552
Epoch 5: -0.2782
Epoch 6: -0.3069
Epoch 7: -0.3331
Epoch 8: -0.3521
Epoch 9: -0.3659
Epoch 10: -0.3747
Epoch 11: -0.3860
Epoch 12: -0.3920
Epoch 13: -0.3972
Epoch 14: -0.3976
Epoch 15: -0.4032
Epoch 16: -0.4093
Epoch 17: -0.4122
Epoch 18: -0.4153
Epoch 19: -0.4176
Epoch 20: -0.4200
Epoch 21: -0.4247
Epoch 22: -0.4274
Epoch 23: -0.4246
Epoch 24: -0.4252
Epoch 25: -0.4303
Epoch 26: -0.4330
Epoch 27: -0.4314
Epoch 28: -0.4353
Epoch 29: -0.4361
Epoch 30: -0.4361
Epoch 31: -0.4374
Epoch 32: -0.4388
Epoch 33: -0.4386
Epoch 34: -0.4398
Epoch 35: -0.4386


In [28]:
X

tensor([[[ 3.8315e+00, -3.9060e+00,  3.8732e+00, -1.0715e+00,  3.9337e+00,
          -4.4028e+00,  4.0799e+00, -1.2841e+00,  1.9892e+00, -1.8131e+00,
           1.8579e+00, -1.1425e+00,  1.5243e+00,  2.1951e-01,  1.0950e+00,
           1.1550e-01, -2.4328e-01, -4.4120e-03, -2.6538e-01, -1.8135e-02,
          -1.2523e+00, -1.0500e-02, -1.1805e+00, -8.3288e-03, -4.4317e-01,
          -2.6055e-03, -4.7139e-01, -1.1309e-02, -4.2580e-01, -3.9368e-02,
          -4.5669e-01, -7.4323e-02, -7.1904e-01, -3.3217e-02, -6.6864e-01,
          -2.0241e-01, -5.8278e-01, -1.8268e-02, -5.9954e-01, -2.0907e-02,
           1.2954e+01, -3.9816e+00,  1.1016e+01, -6.2124e-01,  1.0112e-02,
           1.6653e-01, -1.3166e-01,  2.8125e-01,  5.3892e+00, -4.0361e+00,
           5.0072e+00, -1.2998e+00,  3.7763e+00, -3.5592e+00,  3.8220e+00,
          -1.3662e+00,  1.0792e+00, -1.2481e-01,  7.2540e-01, -1.2112e-01,
           1.4581e+00,  4.9409e-01,  1.0340e+00,  3.0124e-01,  1.6055e+00,
          -1.3379e+00,  1

In [17]:
len(dataset)

817

In [ ]:
dataset[812][0].unsqueeze(1)  

'''
Unsqueeze is done because as the model is trained as batchsize, sequencelen, features and we
have a single sequence as input, we have to unsqueeze it to add another dimension
'''

tensor([[[ 1.7381e+00,  6.3399e-02,  1.3353e+00,  3.9204e-03,  4.3819e-01,
          -4.6187e-01,  3.5383e-01, -8.3247e-01,  1.0054e+00,  7.8383e-01,
           7.8440e-01,  6.6677e-01, -5.8804e-01, -9.0976e-02, -6.4274e-01,
          -3.2847e-01, -2.4067e-01,  7.2490e-03, -2.6279e-01,  3.2253e-02,
          -1.0917e+00, -2.3474e-03, -1.0417e+00,  6.1484e-02, -4.5958e-01,
           1.2524e-02, -4.8698e-01,  5.7144e-02, -2.0942e-01,  3.1706e-02,
          -2.5102e-01,  8.3706e-02, -5.6607e-01,  1.1879e-02, -5.3857e-01,
           9.0565e-04, -3.2978e-01, -1.8268e-02, -3.7098e-01, -2.0907e-02,
           1.5941e+00,  1.0856e+00,  1.3856e+00,  8.9793e-01,  1.2192e-01,
           3.6911e-01, -2.1718e-03,  5.4806e-01,  6.2084e+00,  6.1012e+00,
           6.3469e+00,  1.6257e+00,  1.0977e+00, -1.3248e+00,  1.0380e+00,
          -1.1891e+00,  1.7587e+00, -4.0406e-02,  1.2820e+00, -2.7672e-02,
          -4.1232e-02, -7.7821e-01, -6.1762e-03, -1.2426e+00, -3.0258e-01,
          -6.0762e-01, -2

In [34]:
model.eval()

with torch.no_grad():
    preds = model(dataset[817][0].unsqueeze(1))

In [35]:
preds

tensor([0.1353, 0.2434, 0.3906, 0.0290, 0.2016])